In [143]:
import os 
import geopandas as gpd
import pandas as pd
from shapely import wkt
import geopy.distance
from shapely.geometry import Point
from tqdm import tqdm
from datetime import date
import matplotlib.pyplot as plt

In [144]:
# Define the scenario demand settings
# File name of the scenario input file and output file of aggregated
Date = str(date.today())
# Path and file from Demand and Supply at each sector
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '01_raw')
input_file = '\\00_Scenario_BaseCase_Final.xlsx'  
full_input_path = os.path.abspath(os.path.join(os.getcwd(), input_file_path + input_file))

topo_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
topo_file = '\\input_network_data.xlsx'
full_topo_path = os.path.abspath(os.path.join(os.getcwd(), topo_file_path + topo_file))

output_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
output_file = '\\Demand_Nodes_'+Date+'.csv'  
full_output_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path + output_file))


In [145]:
#Setting for Script 
Save_Output = False
Visualisation = True

In [146]:
#Binary Scenario Settings
#Industry_overall
Industry = True

#Demand sectors
Refineries = False
Chemicals = False
Steel = False 
Paper = False
Mineral_Processing = False 
Metal_Processing = False
Non_Metallic_Minerals = False

Other_Industry = False

Residential_Heat = False 
District_Heat = True

Passenger = False  
Public = True

Air = True
Train = True
Trucks = True

In [147]:
# Functions

def read_data(file_name, sheet_name):
    file = pd.read_excel(file_name, sheet_name=sheet_name)
    file_gpd = gpd.GeoDataFrame(file, geometry=gpd.points_from_xy(file.Longitude, file.Latitude),
                                crs='EPSG:4326').to_crs('EPSG:3857')
    return (file_gpd)


def AggregatedDemand(df1, df2):
    for index, row1 in tqdm(df1.iterrows(), total=df1.shape[0]):
        for _, row2 in df2.iterrows():
            if row1['ID'] == row2['Closest_node']:
                df1.loc[index, 'Demand'] += row2['Peak Load [MWh/h]']
    return df1

def AggregatedSupply(df1, df2):
    for index, row1 in tqdm(df1.iterrows(), total=df1.shape[0]):
        for _, row2 in df2.iterrows():
            if row1['ID'] == row2['Closest_node']:
                df1.loc[index, 'Supply'] += row2['Peak Load [MWh/h]']
    return df1

def AddStorage(df1, df2, aggregated):
    if sum(df2['Peak Load [MWh/h]']) <= 0:
        df2['Peak Load [MWh/h]'] = df2['Peak Load [MWh/h]'] * (-1)
        aggregated = AggregatedSupply(df1, df2)
    else:
        aggregated = AggregatedDemand(df1, df2)

    return aggregated


def find_closest_location(df1, df2):
    closest_node = []
    closest_distance = []

    for _, row1 in tqdm(df1.iterrows(), total=df1.shape[0]):
        max_distance = float('inf')

        for _, row2 in df2.iterrows():
            coords_1 = (row1['Latitude'], row1['Longitude'])
            coords_2 = (row2['Latitude'], row2['Longitude'])
            distance = geopy.distance.distance(coords_1, coords_2).km
            if distance < max_distance:
                node = row2['ID']
                max_distance = distance

        closest_node.append(node)
        closest_distance.append(max_distance)
    return closest_node, closest_distance

def get_NUT3_centroid(df1, df2):
    shape= pd.DataFrame()
    shape['NUTS3'] = [df2.NUTS_ID.values[i] for i in range(len(df2.index))]
    shape['geometry']= [df2.geometry.values[i] for i in range(len(df2.index))]

    merged_df = pd.merge(df1, shape[['NUTS3', 'geometry']], on = 'NUTS3', how='left')
    merged_gdf = gpd.GeoDataFrame(merged_df, geometry= 'geometry',crs = 'EPSG:4326')
    merged_gdf.to_crs('EPSG:3857', inplace = True)
    merged_gdf.dropna(inplace=True)
    merged_gdf['geometry']= [merged_gdf.geometry.centroid.values[i] for i in range(len(merged_gdf.index))]
    merged_gdf.to_crs('EPSG:4326', inplace = True)

    merged_gdf['Longitude']=merged_gdf.geometry.x
    merged_gdf['Latitude']=merged_gdf.geometry.y
    return(merged_gdf)

In [148]:
topo_file = pd.read_excel(full_topo_path)
source_df = topo_file[['source_name', 'source']].drop_duplicates().rename(columns={'source_name':'node_id', 'source':'geometry'})
target_df = topo_file[['target_name', 'target']].drop_duplicates().rename(columns={'target_name':'node_id', 'target':'geometry'})
nodes = pd.concat([source_df, target_df], ignore_index=True)

#nodes['geometry'] = nodes['geometry'].apply(wkt.loads)

# change above otherwise wkt does not work
def tuple_to_wkt_point(s):
    x_str, y_str = s.strip().replace("(", "").replace(")", "").split(",")
    return f"POINT({x_str} {y_str})"

nodes['geometry'] = (nodes['geometry'].astype(str).apply(tuple_to_wkt_point).apply(wkt.loads))

nodes_gdf = gpd.GeoDataFrame(nodes, crs='EPSG:3857')
nodes_gdf.to_crs('EPSG:4326', inplace = True)
nodes_gdf['Longitude'] = nodes_gdf.geometry.apply(lambda p: p.x)
nodes_gdf['Latitude'] = nodes_gdf.geometry.apply(lambda p: p.y)
nodes_gdf['Supply'] = 0
nodes_gdf['Demand'] = 0 

**Code to run the agregated functions for the raw data**

In [149]:
def read_sector(full_path, sheet):
    raw = pd.read_excel(full_path, sheet_name=sheet, header=None)

    # locate the row where the first cell == 'SiteId'
    header_row_idx = raw[raw.iloc[:, 0] == 'SiteId'].index
    header_row = header_row_idx[0]

    # skip everything before that header
    df = pd.read_excel(full_path, sheet_name=sheet, skiprows=header_row)

    # normalise column names
    df.columns = df.columns.str.strip()

    # rows for coordinates 
    df = df.dropna(subset=['Latitude', 'Longitude'])

    # convert commas
    for col in ['Latitude', 'Longitude', 'Peak Load [MWh/h]']:
        if df[col].dtype == object:
            df[col] = df[col].str.replace(',', '.', regex=False).astype(float)

    # make GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.Longitude, df.Latitude),
        crs='EPSG:4326'
    )
    return gdf

In [150]:
# Use defined sectors to test

# AggregateDemand function requires 'ID' column name
nodes_gdf.rename(columns={'node_id': 'ID'}, inplace=True)

sheet_flags = {
    'Refineries': Refineries,
    'Chemicals': Chemicals,
    'Steel': Steel,
    'Paper': Paper,
    'Mineral_Processing': Mineral_Processing,
    'Metal_Processing': Metal_Processing,
    'Non_Metallic_Minerals': Non_Metallic_Minerals,
    'Other_Industry': Other_Industry,
    'Residential Heat': Residential_Heat,
    'District Heat': District_Heat,
    'Passenger': Passenger,
    'Public': Public,
    'Air': Air,
    'Train': Train,
    'Trucks': Trucks
}

# Take the ones with True as boolean value
active_sheets = [s for s, flag in sheet_flags.items() if flag]
active_sheets

['District Heat', 'Public', 'Air', 'Train', 'Trucks']

In [ ]:
for sheet in active_sheets:
    try:
        # Get demand and supply data
        gdf = read_sector(full_input_path, sheet)

        # Debug error if empty 
        if gdf.empty or 'Peak Load [MWh/h]' not in gdf.columns:
            print(f"sheet '{sheet}' empty or missing 'Peak Load [MWh/h]'")
            continue

        # right geopy coordinates
        if gdf.crs != 'EPSG:4326':
            gdf = gdf.to_crs('EPSG:4326')

        # add the lat and long columns
        gdf['Latitude']  = gdf.geometry.y
        gdf['Longitude'] = gdf.geometry.x

        # find closest node wih function
        gdf['Closest_node'], gdf['Distance_km'] = find_closest_location(gdf, nodes_gdf)

        # aggregate demand to nodes
        nodes_gdf = AggregatedDemand(nodes_gdf, gdf) 


    # show the errors 
    except ValueError as e:
        print(f"Could not read sheet '{sheet}': {e}")
    except Exception as e:
        print(f"Error while processing '{sheet}': {e}")

Error while processing 'District Heat': index 0 is out of bounds for axis 0 with size 0
Error while processing 'Public': index 0 is out of bounds for axis 0 with size 0
Error while processing 'Air': index 0 is out of bounds for axis 0 with size 0
Error while processing 'Train': index 0 is out of bounds for axis 0 with size 0
Error while processing 'Trucks': index 0 is out of bounds for axis 0 with size 0


In [152]:
nodes_gdf

,ID,geometry,Longitude,Latitude,Supply,Demand
0,node_0001,POINT (6.80857 54.19578),6.808570,54.195780,0,0
1,node_0002,POINT (7.25666 53.94168),7.256660,53.941680,0,0
2,node_0003,POINT (6.12347 51.80289),6.123468,51.802895,0,0
3,node_0004,POINT (5.72234 51.84537),5.722341,51.845368,0,0
4,node_0005,POINT (6.12728 51.86375),6.127278,51.863752,0,0
...,...,...,...,...,...,...
1016,node_0655,POINT (12.94595 49.76038),12.945950,49.760375,0,0
1017,node_0656,POINT (14.34226 53.8271),14.342265,53.827100,0,0
1018,node_0657,POINT (6.38615 52.47225),6.386150,52.472250,0,0
1019,node_0658,POINT (2.46907 58.18814),2.469066,58.188137,0,0


In [153]:
if Visualisation:
    import plotly.graph_objects as go
    import numpy as np

    df_plot = pd.DataFrame({
        'ID': nodes_gdf['ID'],
        'Longitude': nodes_gdf['Longitude'],
        'Latitude': nodes_gdf['Latitude'],
        'Demand': nodes_gdf['Demand']
    })

    # need to scale the demand for the points size on map 
    df_plot['ScaledDemand'] = df_plot['Demand'].apply(lambda d: np.log1p(d)) + 0.1
    df_plot['Used'] = df_plot['Demand'] > 0


    # Plot the map to visualize
    fig = go.Figure()

    for _, row in df_plot.iterrows():
        fig.add_trace(go.Scattergeo(
            lon=[row['Longitude']],
            lat=[row['Latitude']],
            mode='markers',
            marker=dict(
                size=max(row['ScaledDemand']*5, 2),
                color='red' if row['Used'] else 'blue',
                line=dict(width=0.5, color='black'),
                opacity=0.7
            ),
            hovertext=f"ID: {row['ID']}<br>Demand: {row['Demand']:.2f}",
            showlegend=False
        ))

    fig.update_layout(
        geo=dict(
            scope='europe',
            projection_type='natural earth',
            center=dict(lat=51.2, lon=10.4),
            lataxis=dict(range=[47, 55]),
            lonaxis=dict(range=[5, 16]),
            resolution=50
        ),
        margin={"r":0,"t":50,"l":0,"b":0},
        width=800,
        height=700,
    )

    fig.show()
